# $R_y(\theta)$ over $\{H,\,X,\,Z,\,\mathrm{CCX}\}$

Draws a random angle, synthesises an approximate $R_y(\theta)$ to a requested
precision $\varepsilon$, and returns a gate sequence containing **only** gates from
that set, acting on 1 data qubit (`q0`) + 2 clean ancillas (`q1`, `q2`) that start
and end in $|00\rangle$.

Requires `rysynth.py` next to this notebook.

In [1]:
import math, random, sys, pathlib
import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd()))   # rysynth.py lives next to the notebook
import rysynth as R

print('rysynth loaded from', R.__file__)

rysynth loaded from /home/dustinseboldt/Desktop/iqpe/h+toff synthesis/rysynth.py


## Input

`SEED = None` draws a fresh angle each run; set an integer to reproduce one.
`EPS` is the requested operator-norm precision.

In [2]:
SEED = None          # e.g. 1234 for a reproducible angle
EPS  = 1e-1          # requested precision

rng   = random.Random(SEED)
THETA = rng.uniform(0.0, 4.0 * math.pi)      # R_y has period 4*pi

print('theta = %.12f rad  (%.6f deg)' % (THETA, math.degrees(THETA)))
print('eps   = %.1e' % EPS)

theta = 8.190961584551 rad  (469.307529 deg)
eps   = 1.0e-01


## Synthesis

In [3]:
import time
t0  = time.time()
res = R.synthesize_ry(THETA, EPS)
dt  = time.time() - t0

gates = res['gates']
print('done in %.2f s' % dt)
print('lde k          :', res['k'], ' (~3*log2(1/eps) = %.1f)' % (3*math.log2(1/EPS)))
print('gate count     :', res['counts'])
print('rounded column : v0 =', res['v0'])
print('                 v1 =', res['v1'])

done in 0.02 s
lde k          : 10  (~3*log2(1/eps) = 10.0)
gate count     : {'X': 44, 'H': 12, 'CCX': 45, 'total': 101}
rounded column : v0 = [-17, -27, 2, 1, 1, 0, 0, 0]
                 v1 = [27, -17, -1, 2, 0, 1, 0, 0]


## Output: the gate sequence

`gates` is a list of tuples: `('H', t)`, `('X', t)`, `('Z', t)`, `('CCX', c1, c2, t)`
with qubit indices `0,1,2` and time order left to right (first gate applied first).

In [4]:
def show(gates, head=30):
    for i, g in enumerate(gates[:head]):
        args = ', '.join('q%d' % q for q in g[1:])
        print('%4d  %-4s %s' % (i, g[0], args))
    if len(gates) > head:
        print('      ...  (%d more gates)' % (len(gates) - head))

show(gates)

   0  X    q2
   1  X    q1
   2  X    q0
   3  H    q2
   4  CCX  q0, q1, q2
   5  H    q2
   6  X    q2
   7  X    q1
   8  X    q0
   9  CCX  q0, q1, q2
  10  X    q2
  11  CCX  q0, q2, q1
  12  X    q2
  13  CCX  q0, q1, q2
  14  H    q2
  15  X    q2
  16  CCX  q1, q2, q0
  17  X    q2
  18  X    q0
  19  CCX  q0, q2, q1
  20  X    q0
  21  H    q1
  22  X    q2
  23  CCX  q0, q2, q1
  24  CCX  q1, q2, q0
  25  CCX  q0, q2, q1
  26  X    q2
  27  X    q1
  28  CCX  q0, q1, q2
  29  X    q1
      ...  (71 more gates)


In [5]:
# one-line form
print(R.print_circuit(gates, 25))

X(q2) X(q1) X(q0) H(q2) CCX(q0,q1,q2) H(q2) X(q2) X(q1) X(q0) CCX(q0,q1,q2) X(q2) CCX(q0,q2,q1) X(q2) CCX(q0,q1,q2) H(q2) X(q2) CCX(q1,q2,q0) X(q2) X(q0) CCX(q0,q2,q1) X(q0) H(q1) X(q2) CCX(q0,q2,q1) CCX(q1,q2,q0) ... (+76 more)


In [6]:
# OpenQASM 2.0 export (only h, x, z, ccx appear)
def to_qasm(gates, name='ry_approx'):
    out = ['OPENQASM 2.0;', 'include "qelib1.inc";', 'qreg q[3];']
    for g in gates:
        if g[0] == 'CCX':
            out.append('ccx q[%d],q[%d],q[%d];' % g[1:])
        else:
            out.append('%s q[%d];' % (g[0].lower(), g[1]))
    return '\n'.join(out)

qasm = to_qasm(gates)
pathlib.Path('ry_approx.qasm').write_text(qasm)
print('\n'.join(qasm.splitlines()[:12]))
print('...')
print('written to ry_approx.qasm  (%d lines)' % len(qasm.splitlines()))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
x q[2];
x q[1];
x q[0];
h q[2];
ccx q[0],q[1],q[2];
h q[2];
x q[2];
x q[1];
x q[0];
...
written to ry_approx.qasm  (104 lines)


## Achieved precision

The circuit matrix is rebuilt in exact integer arithmetic as $M/\sqrt2^{\,k}$ and
compared against $R_y(\theta)$ on the clean-ancilla subspace:
$\sup_{\||\psi\rangle\|=1}\ \big\|\,C(|\psi\rangle|00\rangle)-(R_y(\theta)|\psi\rangle)|00\rangle\,\big\|$.

In [7]:
M, k = R.circuit_matrix(gates)                 # exact: integer M, lde k
U    = np.array([[M[i][j] / math.sqrt(2)**k for j in range(8)] for i in range(8)])

err  = R.clean_ancilla_error(gates, THETA)
leak = math.sqrt(sum(U[i,0]**2 + U[i,1]**2 for i in range(2,8)))

print('requested eps    : %.3e' % EPS)
print('achieved error   : %.3e   (%.1f%% of budget)' % (err, 100*err/EPS))
print('ancilla leakage  : %.3e' % leak)
print('within tolerance :', err <= EPS)

used = sorted({g[0] for g in gates})
print('gate set used    :', used, '-> legal:', set(used) <= {'H','X','Z','CCX'})
print('qubits used      :', sorted({q for g in gates for q in g[1:]}))

requested eps    : 1.000e-01
achieved error   : 9.424e-02   (94.2% of budget)
ancilla leakage  : 1.083e-01
within tolerance : True
gate set used    : ['CCX', 'H', 'X'] -> legal: True
qubits used      : [0, 1, 2]


In [8]:
# side-by-side check of the 2x2 block
c, s = math.cos(THETA/2), math.sin(THETA/2)
print('synthesised block      ideal R_y(theta)')
for i in range(2):
    print('[% .9f % .9f]   [% .9f % .9f]'
          % (U[i,0], U[i,1], [c, s][i], [-s, c][i]))

synthesised block      ideal R_y(theta)
[-0.531250000  0.843750000]   [-0.578516028  0.815671015]
[-0.843750000 -0.531250000]   [-0.815671015 -0.578516028]


In [9]:
# full simulation on a random input state
psi = np.array([rng.gauss(0,1), rng.gauss(0,1)]); psi /= np.linalg.norm(psi)
inp = np.zeros(8); inp[0], inp[1] = psi          # |psi> (x) |00>, q0 = LSB
out = U @ inp
want = np.array([c*psi[0] - s*psi[1], s*psi[0] + c*psi[1]])
print('input |psi>        :', np.round(psi, 8))
print('circuit output     :', np.round(out[:2], 9), ' ancillas:', np.round(out[2:], 9))
print('R_y(theta)|psi>    :', np.round(want, 9))
print('||difference||     : %.3e' % np.linalg.norm(out[:2] - want))

input |psi>        : [0.99920087 0.03997025]
circuit output     : [-0.49710056 -0.86430993]  ancillas: [0.06120098 0.03372317 0.03122503 0.00124907 0.         0.        ]
R_y(theta)|psi>    : [-0.54545114 -0.83814262]
||difference||     : 5.498e-02


## Batch: several random angles

In [10]:
print(' theta        eps      k   gates   CCX     H     X    achieved   ok')
print('-'*72)
for _ in range(6):
    th = rng.uniform(0, 4*math.pi)
    r  = R.synthesize_ry(th, EPS)
    e  = R.clean_ancilla_error(r['gates'], th)
    c_ = r['counts']
    print('%8.5f  %7.1e  %3d  %5d  %5d %5d %5d   %.2e   %s'
          % (th, EPS, r['k'], c_['total'], c_.get('CCX',0),
             c_.get('H',0), c_.get('X',0), e, e <= EPS))

 theta        eps      k   gates   CCX     H     X    achieved   ok
------------------------------------------------------------------------
 5.71041  1.0e-01    9     82     39    13    30   9.91e-02   True
 4.36091  1.0e-01   10     88     46    14    28   9.02e-02   True
 5.38908  1.0e-01   11     99     58    15    26   6.89e-02   True
 3.97190  1.0e-01   10     85     43    14    28   8.36e-02   True
 9.94959  1.0e-01    9     84     41    11    32   8.59e-02   True
 5.54616  1.0e-01   10    101     47    14    40   5.68e-02   True


## Notes

* `q1`, `q2` are **clean** ancillas: they start in $|00\rangle$ and return to it up to
  $O(\varepsilon)$ (the `ancilla leakage` figure above), so the circuit is a genuine
  drop-in gate, not a catalytic one.
* $R_x(\theta)$ is **not** reachable this way. Any real matrix is at least
  $|\sin(\theta/2)|$ away from $R_x(\theta)$, so it needs a permanent extra qubit
  encoding $i$.
* Runtime grows like $\varepsilon^{-1/2}$ from the linear scan over $p$; below
  $\varepsilon\approx10^{-11}$ expect tens of seconds per call.
* Angles are reduced mod $4\pi$, not $2\pi$: $R_y(\theta+2\pi) = -R_y(\theta)$, and the
  sign is a real global phase that this construction tracks exactly.